# Mixed Follow-Up Expanded-Data Rerun Audit

during the main experiments of moe study project, i noticed one validity concern, that some approaches trained on more than one epoch.
We later decided that we should take the original dataset, expand the dataset while preserving the language distribution and retraining the approaches taht trained on more than one epoch on this dataset.
This is exactly what i have done. In this notebok i want to go throught the processa and check and verify that all the steps worked properly.

specifically, i want to checks four things:
- the original mixed follow-up had 7 runs with `epoch > 1` (recreating / showing original problem)
- the copied dataset was expanded to 400k train chunks while preserving the language distribution and validation split
- the 7 rerun approaches now stay below one epoch under the same 7.6h training setup
- what changes in the results? do the approaches with the rerun change their ranking and promotion decision (we prompted the best runs to the next stage of the experiment)


In [8]:
# first, these are the paths and data we need
from pathlib import Path
import re
import sys

import numpy as np
import pandas as pd
from datasets import load_from_disk
from IPython.display import Markdown, display
from scipy.stats import friedmanchisquare, wilcoxon

ROOT = next(
    path for path in [Path.cwd(), *Path.cwd().parents]
    if (path / "configs/pipeline").exists() and (path / "outputs/pipeline_runs").exists()
)

for path in [ROOT / "scripts/analysis", ROOT / "result_analysis/notebooks/validation"]:
    if str(path) not in sys.path:
        sys.path.insert(0, str(path))

from mixed_followup_dataset_utils import dataset_distribution, read_json as read_json_file, strip_zero_counts
from util import load_training_metrics, read_json as read_artifact_json

ORIGINAL_REPORT = ROOT / "outputs/pipeline_runs/moe_study_expert_sweep_pipeline_report/mix_follow_up_report.xlsx"
ORIGINAL_CONSOLIDATED_ROOT = ROOT / "outputs/pipeline_runs/moe_study_mixed_rank_expert_followup_consolidated_current_report"

DATA_RERUN_ROOT = ROOT / "outputs/pipeline_runs/moe_study_mixed_followup_expanded_data_rerun_10_langs"
OLD_TOKENIZED = DATA_RERUN_ROOT / "tokenized"
EXPANDED_TOKENIZED = DATA_RERUN_ROOT / "tokenized_expanded_400k_train"
AUDIT_DIR = DATA_RERUN_ROOT / "tokenized_expanded_400k_train_audit"

RERUN_OUTPUT_ROOT = ROOT / "outputs/pipeline_runs/moe_study_mixed_rank_expert_followup_expanded_data_rerun_noctua2_a100_4gpu_10_langs_7p6h"
RERUN_REPORT = RERUN_OUTPUT_ROOT / "moe_study_mixed_rank_expert_followup_expanded_data_rerun_noctua2_a100_4gpu_10_langs_7p6h.xlsx"

CORRECTED_REPORT_CANDIDATES = [
    ROOT / "moe_study_mix_follow_up_study_results_corrected_expanded_data_26.06.26_v1.xlsx",
    ROOT / "outputs/pipeline_runs/moe_study_mix_follow_up_study_results_corrected_expanded_data_26.06.26_v1.xlsx",
]
CORRECTED_REPORT = next((path for path in CORRECTED_REPORT_CANDIDATES if path.exists()), None)

EXPECTED_RERUNS = {
    "hydralora_r4_heads2",
    "lora_r4",
    "moelora_tr4_e2_k1",
    "moelora_tr8_e2_k1",
    "mola_tr4_e2_k1",
    "mola_tr8_e2_k1",
    "mov_dense_e2",
}
PROMOTED_TO_FINAL = {
    "hydralora_r2_heads32",
    "moelora_tr64_e32_k4",
    "mola_tr64_e32_k4",
}
EPOCH_REPEAT_THRESHOLD = 1.0001

paths = pd.DataFrame([
    {"artifact": "original mixed follow-up report", "path": ORIGINAL_REPORT, "exists": ORIGINAL_REPORT.exists()},
    {"artifact": "old copied tokenized dataset", "path": OLD_TOKENIZED, "exists": OLD_TOKENIZED.exists()},
    {"artifact": "expanded tokenized dataset", "path": EXPANDED_TOKENIZED, "exists": EXPANDED_TOKENIZED.exists()},
    {"artifact": "expanded dataset audit", "path": AUDIT_DIR, "exists": AUDIT_DIR.exists()},
    {"artifact": "7-run expanded-data rerun report", "path": RERUN_REPORT, "exists": RERUN_REPORT.exists()},
    {"artifact": "corrected combined report", "path": CORRECTED_REPORT or "not present; synthesized below", "exists": CORRECTED_REPORT is not None},
])
display(paths)


,artifact,path,exists
0,original mixed follow-up report,/scratch/hpc-prf-merlin/joel/moe-study/outputs...,True
1,old copied tokenized dataset,/scratch/hpc-prf-merlin/joel/moe-study/outputs...,True
2,expanded tokenized dataset,/scratch/hpc-prf-merlin/joel/moe-study/outputs...,True
3,expanded dataset audit,/scratch/hpc-prf-merlin/joel/moe-study/outputs...,True
4,7-run expanded-data rerun report,/scratch/hpc-prf-merlin/joel/moe-study/outputs...,True
5,corrected combined report,not present; synthesized below,False


## 1. Reproduce the Original Exposure Issue

The original mixed follow-up validity notebook identified repeated-data exposure from the training artifacts. We repeat that check here by reading each run's `train_results.json` and `trainer_state.json` through the same validation helper.

In [2]:
def require_paths(*paths):
    missing = [path for path in paths if isinstance(path, Path) and not path.exists()]
    if missing:
        raise FileNotFoundError("Missing required artifact(s): " + ", ".join(str(path) for path in missing))


def truthy(series):
    return series.fillna(False).astype(str).str.lower().isin(["true", "1", "yes"])


def load_run_sources(consolidated_root):
    profile = read_artifact_json(consolidated_root / "state/resolved_profile.json")
    return profile.get("consolidated_report_sources", {}).get("run_sources", {})


def audit_training_exposure(report_path, *, output_root=None, consolidated_root=None):
    require_paths(report_path)
    raw = pd.read_excel(report_path, sheet_name="benchmark_raw")
    non_baseline = raw[~truthy(raw["is_baseline"])].copy()
    run_sources = load_run_sources(consolidated_root) if consolidated_root else {}

    rows = []
    for run_id, group in sorted(non_baseline.groupby("run_id")):
        source_root = Path(output_root) if output_root else None
        if source_root is None and str(run_id) in run_sources:
            source_root = ROOT / "outputs/pipeline_runs" / str(run_sources[str(run_id)])
        run_dir = source_root / "runs" / str(run_id) if source_root else None
        row = {
            "run_id": str(run_id),
            "kind": str(group["kind"].iloc[0]),
            "source_root": str(source_root.relative_to(ROOT)) if source_root and source_root.is_relative_to(ROOT) else str(source_root or ""),
            "run_dir_exists": bool(run_dir and run_dir.exists()),
        }
        row.update(load_training_metrics(run_dir))
        rows.append(row)

    out = pd.DataFrame(rows)
    for col in ["epoch", "num_train_epochs", "global_step", "max_steps", "train_runtime", "num_input_tokens_seen"]:
        if col in out:
            out[col] = pd.to_numeric(out[col], errors="coerce")
    out["actual_repeat"] = out["epoch"] > EPOCH_REPEAT_THRESHOLD
    out["configured_repeat_risk"] = out["num_train_epochs"] > 1
    out["train_hours"] = out["train_runtime"] / 3600
    return out.sort_values("run_id").reset_index(drop=True)


old_exposure = audit_training_exposure(ORIGINAL_REPORT, consolidated_root=ORIGINAL_CONSOLIDATED_ROOT)
old_repeat = old_exposure[old_exposure.actual_repeat].copy()

display(old_exposure.groupby(["actual_repeat", "configured_repeat_risk"]).agg(
    runs=("run_id", "nunique"),
    epoch_min=("epoch", "min"),
    epoch_max=("epoch", "max"),
    hours_median=("train_hours", "median"),
).reset_index())
display(old_repeat[["run_id", "kind", "epoch", "global_step", "max_steps", "train_hours", "source_root"]])

repeat_set = set(old_repeat.run_id)
display(Markdown(
    f"Original repeated-exposure runs: `{len(repeat_set)}`. "
    f"They match the expected 7 rerun targets: `{repeat_set == EXPECTED_RERUNS}`."
))
assert repeat_set == EXPECTED_RERUNS, sorted(repeat_set ^ EXPECTED_RERUNS)


,actual_repeat,configured_repeat_risk,runs,epoch_min,epoch_max,hours_median
0,False,True,7,0.432606,0.954534,7.509326
1,True,True,7,1.471715,3.295367,7.489769


,run_id,kind,epoch,global_step,max_steps,train_hours,source_root
2,hydralora_r4_heads2,hydralora,1.471715,3049,20000,7.404099,outputs/pipeline_runs/moe_study_mixed_rank_exp...
3,lora_r4,lora,2.518989,5219,20000,7.539885,outputs/pipeline_runs/moe_study_mixed_rank_exp...
6,moelora_tr4_e2_k1,moelora,2.136155,4426,20000,7.470315,outputs/pipeline_runs/moe_study_mixed_rank_exp...
9,moelora_tr8_e2_k1,moelora,2.141466,4437,20000,7.582227,outputs/pipeline_runs/moe_study_mixed_rank_exp...
10,mola_tr4_e2_k1,mola,1.792999,2476,20000,7.489769,outputs/pipeline_runs/moe_study_mixed_rank_exp...
12,mola_tr8_e2_k1,mola,1.792275,2475,20000,7.514700,outputs/pipeline_runs/moe_study_mixed_rank_exp...
13,mov_dense_e2,movlora,3.295367,1707,20000,7.456094,outputs/pipeline_runs/moe_study_mixed_rank_exp...


Original repeated-exposure runs: `7`. They match the expected 7 rerun targets: `True`.

## 2. Verify the Expanded Dataset

The old dataset was copied into the expanded-data rerun output directory before appending new samples. The expansion target was 400k train chunks while preserving the old language distribution and expanding validation with the same rules. The source-level audit must also show no overlap between old and newly appended source keys.

In [3]:
require_paths(OLD_TOKENIZED, EXPANDED_TOKENIZED, AUDIT_DIR)

target = read_json_file(AUDIT_DIR / "expanded_distribution_target.json")
actual = read_json_file(AUDIT_DIR / "expanded_distribution_actual.json")
newness = read_json_file(AUDIT_DIR / "newness_validation.json")

old_counted = dataset_distribution(load_from_disk(str(OLD_TOKENIZED)), include_doc_counts=False)
expanded_counted = dataset_distribution(load_from_disk(str(EXPANDED_TOKENIZED)), include_doc_counts=False)
expected_splits = {"train": 400_000, "validation": 7_705}

split_summary = pd.DataFrame([
    {
        "dataset": "old copied",
        "rows": old_counted["rows"],
        "train": old_counted["splits"].get("train", 0),
        "validation": old_counted["splits"].get("validation", 0),
        "duplicate_doc_chunk_keys": old_counted["duplicate_doc_chunk_keys"],
    },
    {
        "dataset": "expanded",
        "rows": expanded_counted["rows"],
        "train": expanded_counted["splits"].get("train", 0),
        "validation": expanded_counted["splits"].get("validation", 0),
        "duplicate_doc_chunk_keys": expanded_counted["duplicate_doc_chunk_keys"],
    },
])
display(split_summary)

def language_distribution_table(old_dist, expanded_dist, target):
    old_train = old_dist["samples_per_language_by_split"].get("train", {})
    old_val = old_dist["samples_per_language_by_split"].get("validation", {})
    new_train = expanded_dist["samples_per_language_by_split"].get("train", {})
    new_val = expanded_dist["samples_per_language_by_split"].get("validation", {})
    target_train = target["target_train_by_language"]
    target_val = target["target_validation_by_language"]
    languages = sorted(set(old_train) | set(old_val) | set(new_train) | set(new_val) | set(target_train) | set(target_val))
    old_train_total = sum(old_train.values())
    old_val_total = sum(old_val.values())
    new_train_total = sum(new_train.values())
    new_val_total = sum(new_val.values())
    rows = []
    for language in languages:
        rows.append({
            "language": language,
            "old_train": int(old_train.get(language, 0)),
            "expanded_train": int(new_train.get(language, 0)),
            "target_train": int(target_train.get(language, 0)),
            "appended_train": int(new_train.get(language, 0)) - int(old_train.get(language, 0)),
            "old_validation": int(old_val.get(language, 0)),
            "expanded_validation": int(new_val.get(language, 0)),
            "target_validation": int(target_val.get(language, 0)),
            "appended_validation": int(new_val.get(language, 0)) - int(old_val.get(language, 0)),
            "old_train_pct": int(old_train.get(language, 0)) / old_train_total,
            "expanded_train_pct": int(new_train.get(language, 0)) / new_train_total,
            "old_validation_pct": int(old_val.get(language, 0)) / old_val_total if old_val_total else 0,
            "expanded_validation_pct": int(new_val.get(language, 0)) / new_val_total if new_val_total else 0,
        })
    out = pd.DataFrame(rows)
    out["train_pct_delta"] = out.expanded_train_pct - out.old_train_pct
    out["validation_pct_delta"] = out.expanded_validation_pct - out.old_validation_pct
    return out

distribution = language_distribution_table(old_counted, expanded_counted, target)
display(distribution)

checks = pd.DataFrame([
    {"check": "old copied split counts match expansion input", "pass": old_counted["splits"] == {"train": target["current_train_chunks"], "validation": target["current_validation_chunks"]}},
    {"check": "expanded split counts are 400k train plus 7705 validation", "pass": expanded_counted["splits"] == expected_splits},
    {"check": "audit split counts match expected", "pass": actual.get("splits") == expected_splits},
    {"check": "train language counts match target", "pass": strip_zero_counts(expanded_counted["samples_per_language_by_split"].get("train", {})) == strip_zero_counts(target["target_train_by_language"])},
    {"check": "validation language counts match target", "pass": strip_zero_counts(expanded_counted["samples_per_language_by_split"].get("validation", {})) == strip_zero_counts(target["target_validation_by_language"])},
    {"check": "no duplicate doc/chunk keys in loaded dataset", "pass": expanded_counted["duplicate_doc_chunk_keys"] == 0},
    {"check": "no duplicate doc/chunk keys in audit", "pass": actual.get("duplicate_doc_chunk_keys") == 0},
    {"check": "new source keys do not overlap old source keys", "pass": int(newness.get("new_vs_old_source_key_overlap", -1)) == 0},
    {"check": "new train source keys do not overlap validation source keys", "pass": int(newness.get("new_train_vs_validation_source_key_overlap", -1)) == 0},
    {"check": "new text hashes do not overlap old train text hashes", "pass": int(newness.get("text_hash_overlaps_with_old", {}).get("train", -1)) == 0},
    {"check": "new text hashes do not overlap old validation text hashes", "pass": int(newness.get("text_hash_overlaps_with_old", {}).get("validation", -1)) == 0},
])
display(checks)
display(Markdown(
    f"Maximum absolute language-ratio drift: train `{distribution.train_pct_delta.abs().max():.6f}`, "
    f"validation `{distribution.validation_pct_delta.abs().max():.6f}`."
))
assert checks["pass"].all(), checks.loc[~checks["pass"], "check"].tolist()


,dataset,rows,train,validation,duplicate_doc_chunk_keys
0,old copied,101325,99410,1915,0
1,expanded,407705,400000,7705,0


,language,old_train,expanded_train,target_train,appended_train,old_validation,expanded_validation,target_validation,appended_validation,old_train_pct,expanded_train_pct,old_validation_pct,expanded_validation_pct,train_pct_delta,validation_pct_delta
0,arb_Arab,2913,11721,11721,8808,46,185,185,139,0.029303,0.029302,0.024021,0.024010,-3.870335e-07,-1.050486e-05
1,arb_Latn,12,48,48,36,0,0,0,0,0.000121,0.000120,0.000000,0.000000,-7.122020e-07,0.000000e+00
2,azj_Cyrl,4,16,16,12,0,0,0,0,0.000040,0.000040,0.000000,0.000000,-2.374007e-07,0.000000e+00
3,azj_Latn,767,3086,3086,2319,13,52,52,39,0.007716,0.007715,0.006789,0.006749,-5.215773e-07,-3.964738e-05
4,cmn_Hani,26857,108066,108066,81209,562,2261,2261,1699,0.270164,0.270165,0.293473,0.293446,1.032592e-06,-2.677045e-05
5,english,2212,8900,8900,6688,47,189,189,142,0.022251,0.022250,0.024543,0.024530,-1.282567e-06,-1.355466e-05
6,fas_Arab,2728,10977,10977,8249,56,226,226,170,0.027442,0.027443,0.029243,0.029332,5.927472e-07,8.878301e-05
7,hun_Latn,3285,13218,13218,9933,85,342,342,257,0.033045,0.033045,0.044386,0.044387,3.470476e-08,3.388665e-07
8,ind_Latn,4114,16554,16554,12440,84,338,338,254,0.041384,0.041385,0.043864,0.043868,8.334172e-07,3.388665e-06
9,jpn_Jpan,15790,63535,63535,47745,287,1155,1155,868,0.158837,0.158837,0.149869,0.149903,3.608792e-07,3.320891e-05


,check,pass
0,old copied split counts match expansion input,True
1,expanded split counts are 400k train plus 7705...,True
2,audit split counts match expected,True
3,train language counts match target,True
4,validation language counts match target,True
5,no duplicate doc/chunk keys in loaded dataset,True
6,no duplicate doc/chunk keys in audit,True
7,new source keys do not overlap old source keys,True
8,new train source keys do not overlap validatio...,True
9,new text hashes do not overlap old train text ...,True


Maximum absolute language-ratio drift: train `0.000001`, validation `0.000089`.

## 3. Verify the Rerun Training Exposure

The 7 targeted approaches were trained again using the expanded dataset. The curcial check now is that these reruns no longer train for more than one epoch.

In [4]:
rerun_exposure = audit_training_exposure(RERUN_REPORT, output_root=RERUN_OUTPUT_ROOT)
display(rerun_exposure[["run_id", "kind", "epoch", "global_step", "max_steps", "train_hours", "run_dir_exists"]])

rerun_checks = pd.DataFrame([
    {"check": "rerun report contains exactly the 7 targeted approaches", "pass": set(rerun_exposure.run_id) == EXPECTED_RERUNS},
    {"check": "all rerun training directories exist", "pass": bool(rerun_exposure.run_dir_exists.all())},
    {"check": "all reruns have training metrics", "pass": bool(rerun_exposure.train_results_present.fillna(False).all() and rerun_exposure.trainer_state_present.fillna(False).all())},
    {"check": "no rerun crosses one epoch", "pass": bool((rerun_exposure.epoch <= 1.0).all())},
])
display(rerun_checks)
display(Markdown(
    f"Rerun runtime range: `{rerun_exposure.train_hours.min():.2f}h` to `{rerun_exposure.train_hours.max():.2f}h`; "
    f"median `{rerun_exposure.train_hours.median():.2f}h`."
))
assert rerun_checks["pass"].all(), rerun_checks.loc[~rerun_checks["pass"], "check"].tolist()


,run_id,kind,epoch,global_step,max_steps,train_hours,run_dir_exists
0,hydralora_r4_heads2,hydralora,0.339840,2832,8333,7.543672,True
1,lora_r4,lora,0.628560,5238,8333,7.516055,True
2,moelora_tr4_e2_k1,moelora,0.510960,4258,8333,7.530264,True
3,moelora_tr8_e2_k1,moelora,0.529440,4412,8333,7.507087,True
4,mola_tr4_e2_k1,mola,0.449991,2500,5555,7.530221,True
5,mola_tr8_e2_k1,mola,0.449631,2498,5555,7.538193,True
6,mov_dense_e2,movlora,0.828414,1726,2083,7.500840,True


,check,pass
0,rerun report contains exactly the 7 targeted a...,True
1,all rerun training directories exist,True
2,all reruns have training metrics,True
3,no rerun crosses one epoch,True


Rerun runtime range: `7.50h` to `7.54h`; median `7.53h`.

## 4. Replace the 7 Results and Recompute the Mixed Follow-Up Ranking

The corrected result table can now be built by keeping all old benchmark results except the 7 repeated-exposure runs, and replacing those 7 run IDs with their expanded-data rerun benchmark rows.

In [9]:
def load_report_raw(path):
    require_paths(path)
    return pd.read_excel(path, sheet_name="benchmark_raw")


old_raw = load_report_raw(ORIGINAL_REPORT)
rerun_raw = load_report_raw(RERUN_REPORT)
rerun_non_baseline = rerun_raw[~truthy(rerun_raw["is_baseline"])].copy()
rerun_ids = set(rerun_non_baseline.run_id.astype(str).unique())

corrected_raw = pd.concat([
    old_raw[~old_raw.run_id.astype(str).isin(rerun_ids)],
    rerun_non_baseline,
], ignore_index=True)

if CORRECTED_REPORT is not None:
    workbook_corrected_raw = load_report_raw(CORRECTED_REPORT)
    key_cols = ["run_id", "kind", "benchmark", "task", "language", "metric"]
    workbook_pairs = old_raw.merge(workbook_corrected_raw, on=key_cols, how="inner", suffixes=("_old", "_corrected"))
    changed = workbook_pairs[
        pd.to_numeric(workbook_pairs.value_old, errors="coerce").ne(pd.to_numeric(workbook_pairs.value_corrected, errors="coerce"))
    ]
    changed_run_ids = set(changed.run_id.astype(str))
else:
    changed_run_ids = rerun_ids

key_cols = ["run_id", "kind", "benchmark", "task", "language", "metric"]
old_rerun_keys = old_raw[old_raw.run_id.astype(str).isin(rerun_ids)][key_cols]
new_rerun_keys = rerun_non_baseline[key_cols]
old_only = old_rerun_keys.merge(new_rerun_keys, on=key_cols, how="left", indicator=True).query("_merge == 'left_only'")
new_only = new_rerun_keys.merge(old_rerun_keys, on=key_cols, how="left", indicator=True).query("_merge == 'left_only'")

replacement_checks = pd.DataFrame([
    {"check": "rerun workbook contains exactly the expected 7 run IDs", "pass": rerun_ids == EXPECTED_RERUNS},
    {"check": "old and rerun workbooks have the same benchmark keys for those 7 runs", "pass": old_only.empty and new_only.empty},
    {"check": "corrected combined report changes only the 7 expected run IDs", "pass": changed_run_ids == EXPECTED_RERUNS},
])

display(pd.DataFrame([
    {"source": "old repeated-exposure rows", "rows": len(old_raw[old_raw.run_id.astype(str).isin(rerun_ids)]), "run_ids": len(rerun_ids)},
    {"source": "expanded-data rerun rows", "rows": len(rerun_non_baseline), "run_ids": len(rerun_ids)},
    {"source": "corrected combined rows", "rows": len(corrected_raw), "run_ids": corrected_raw.run_id.nunique()},
]))
display(replacement_checks)
assert replacement_checks["pass"].all(), replacement_checks.loc[~replacement_checks["pass"], "check"].tolist()


,source,rows,run_ids
0,old repeated-exposure rows,651,7
1,expanded-data rerun rows,651,7
2,corrected combined rows,1395,15


,check,pass
0,rerun workbook contains exactly the expected 7...,True
1,old and rerun workbooks have the same benchmar...,True
2,corrected combined report changes only the 7 e...,True


In [10]:
def primary_mask(df):
    return (
        ((df.benchmark == "Belebele") & (df.metric == "acc_norm")) |
        ((df.benchmark == "FLORES") & (df.metric.astype(str).str.startswith("bits_per_byte "))) |
        ((df.benchmark == "SIB200") & (df.metric == "acc")) |
        ((df.benchmark == "Taxi1500") & (df.metric == "acc"))
    )


def parse_total_rank(run_id):
    run_id = str(run_id)
    match = re.search(r"_tr(\d+)_", run_id)
    if match:
        return int(match.group(1))
    match = re.search(r"_r(\d+)(?:_|$)", run_id)
    return int(match.group(1)) if match else np.nan


def parse_component_count(run_id):
    run_id = str(run_id)
    for pattern in [r"_e(\d+)(?:_|$)", r"_heads(\d+)$", r"_nup(\d+)$", r"_b(\d+)$"]:
        match = re.search(pattern, run_id)
        if match:
            return int(match.group(1))
    return np.nan


def parse_top_k(run_id):
    match = re.search(r"_k(\d+)(?:_|$)", str(run_id))
    return int(match.group(1)) if match else np.nan


def holm(pvals):
    pvals = np.asarray(pd.Series(pvals, dtype=float), dtype=float)
    adjusted = np.full(len(pvals), np.nan, dtype=float)
    valid = np.where(np.isfinite(pvals))[0]
    if len(valid) == 0:
        return adjusted
    order = valid[np.argsort(pvals[valid])]
    raw_adj = np.empty(len(order), dtype=float)
    m = len(order)
    for rank, idx in enumerate(order):
        raw_adj[rank] = min((m - rank) * pvals[idx], 1.0)
    raw_adj = np.maximum.accumulate(raw_adj)
    for idx, value in zip(order, raw_adj):
        adjusted[idx] = value
    return adjusted


def mixed_followup_analysis(raw):
    primary = raw[primary_mask(raw)].copy()
    primary = primary[~truthy(primary["is_baseline"])].copy()
    primary["direction"] = primary.metric.astype(str).str.extract(r"(en->x|x->en)", expand=False).fillna("")
    primary["score_family"] = np.where(primary.benchmark.eq("FLORES"), "FLORES " + primary.direction, primary.benchmark)
    primary["total_rank"] = primary.run_id.map(parse_total_rank)
    primary["component_count"] = primary.run_id.map(parse_component_count)
    primary["top_k"] = primary.run_id.map(parse_top_k)
    primary["value"] = pd.to_numeric(primary.value, errors="coerce")

    coverage = (
        primary.groupby(["run_id", "kind"], dropna=False)
        .agg(primary_rows=("value", "count"), score_families=("score_family", "nunique"), tasks=("task", "nunique"))
        .reset_index()
    )
    expected_rows = int(coverage.primary_rows.max())
    complete_runs = coverage.loc[coverage.primary_rows.eq(expected_rows), "run_id"].astype(str).tolist()

    metrics = primary[primary.run_id.astype(str).isin(complete_runs)].dropna(subset=["value"]).copy()
    g = metrics.groupby(["score_family", "task", "metric"]).value
    lo, hi = g.transform("min"), g.transform("max")
    metrics["normalized_score"] = np.where(
        metrics.better.eq("lower"),
        (hi - metrics.value) / (hi - lo),
        (metrics.value - lo) / (hi - lo),
    )
    metrics.loc[(hi - lo).eq(0), "normalized_score"] = 1.0

    config_summary = (
        metrics.groupby(["run_id", "kind", "total_rank", "component_count", "top_k"], dropna=False)
        .agg(
            mean_score=("normalized_score", "mean"),
            n_rows=("normalized_score", "count"),
            mean_belebele=("normalized_score", lambda s: s[metrics.loc[s.index, "benchmark"].eq("Belebele")].mean()),
            mean_flores=("normalized_score", lambda s: s[metrics.loc[s.index, "benchmark"].eq("FLORES")].mean()),
            mean_sib=("normalized_score", lambda s: s[metrics.loc[s.index, "benchmark"].eq("SIB200")].mean()),
            mean_taxi=("normalized_score", lambda s: s[metrics.loc[s.index, "benchmark"].eq("Taxi1500")].mean()),
        )
        .sort_values("mean_score", ascending=False)
        .reset_index()
    )
    config_summary["mean_score_rank"] = range(1, len(config_summary) + 1)

    pivot = (
        metrics.groupby(["score_family", "task", "language", "metric", "run_id"])["normalized_score"]
        .mean()
        .unstack("run_id")
        .dropna()
    )
    stat, p_raw = friedmanchisquare(*[pivot[column] for column in pivot.columns])
    avg_ranks = pivot.rank(axis=1, ascending=False).mean().sort_values()
    friedman_ranks = avg_ranks.reset_index()
    friedman_ranks.columns = ["run_id", "friedman_avg_rank"]
    friedman_ranks["friedman_order"] = range(1, len(friedman_ranks) + 1)

    ranking = config_summary.merge(friedman_ranks, on="run_id", how="left").sort_values("friedman_order")
    overall = {
        "n_pairs": len(pivot),
        "n_configs": pivot.shape[1],
        "friedman_p_raw": p_raw,
        "kendalls_w": stat / (len(pivot) * (pivot.shape[1] - 1)),
        "top3_configs": ranking.head(3).run_id.tolist(),
        "top4_configs": ranking.head(4).run_id.tolist(),
    }
    return {"metrics": metrics, "coverage": coverage, "ranking": ranking, "overall": overall, "pivot": pivot}


old_analysis = mixed_followup_analysis(old_raw)
corrected_analysis = mixed_followup_analysis(corrected_raw)

old_ranking = old_analysis["ranking"].add_prefix("old_").rename(columns={"old_run_id": "run_id"})
corrected_ranking = corrected_analysis["ranking"].add_prefix("corrected_").rename(columns={"corrected_run_id": "run_id"})
ranking_comparison = old_ranking.merge(corrected_ranking, on="run_id", how="outer")
ranking_comparison["was_rerun"] = ranking_comparison.run_id.isin(EXPECTED_RERUNS)
ranking_comparison["delta_friedman_avg_rank"] = ranking_comparison.corrected_friedman_avg_rank - ranking_comparison.old_friedman_avg_rank
ranking_comparison["delta_mean_score"] = ranking_comparison.corrected_mean_score - ranking_comparison.old_mean_score

display(Markdown("**Old top 10 by Friedman average rank**"))
display(old_analysis["ranking"][["friedman_order", "mean_score_rank", "run_id", "kind", "mean_score", "friedman_avg_rank"]].head(10))
display(Markdown("**Corrected top 10 by Friedman average rank**"))
display(corrected_analysis["ranking"][["friedman_order", "mean_score_rank", "run_id", "kind", "mean_score", "friedman_avg_rank"]].head(10))
display(Markdown("**Rerun impact on the 7 replaced approaches**"))
display(ranking_comparison[ranking_comparison.was_rerun][[
    "run_id", "old_friedman_avg_rank", "corrected_friedman_avg_rank", "delta_friedman_avg_rank",
    "old_mean_score", "corrected_mean_score", "delta_mean_score",
]].sort_values("corrected_friedman_avg_rank"))


**Old top 10 by Friedman average rank**

,friedman_order,mean_score_rank,run_id,kind,mean_score,friedman_avg_rank
2,1,3,moelora_tr64_e32_k4,moelora,0.789081,4.226190
0,2,1,hydralora_r2_heads32,hydralora,0.801961,4.238095
1,3,2,moelora_tr64_e32_k8,moelora,0.799617,4.380952
3,4,4,moelora_tr128_e32_k8,moelora,0.765262,5.214286
4,5,5,mola_tr64_e32_k4,mola,0.721210,5.797619
5,6,6,hydralora_r2_heads2,hydralora,0.708734,6.142857
6,7,7,mov_dense_e2,movlora,0.632994,7.011905
7,8,8,moelora_tr8_e2_k1,moelora,0.584838,7.607143
8,9,9,moelora_tr128_e32_k4,moelora,0.578661,8.095238
10,10,11,mola_tr8_e2_k1,mola,0.553305,8.273810


**Corrected top 10 by Friedman average rank**

,friedman_order,mean_score_rank,run_id,kind,mean_score,friedman_avg_rank
1,1,2,moelora_tr64_e32_k4,moelora,0.633914,5.535714
0,2,1,hydralora_r2_heads32,hydralora,0.657946,5.583333
2,3,3,moelora_tr64_e32_k8,moelora,0.631887,5.869048
3,4,4,moelora_tr8_e2_k1,moelora,0.587054,6.571429
4,5,5,moelora_tr128_e32_k8,moelora,0.585509,6.845238
6,6,7,mola_tr64_e32_k4,mola,0.545069,7.190476
5,7,6,mola_tr8_e2_k1,mola,0.545755,7.547619
7,8,8,mola_tr4_e2_k1,mola,0.522345,7.583333
10,9,11,lora_r4,lora,0.474582,8.047619
9,10,10,moelora_tr4_e2_k1,moelora,0.476779,8.071429


**Rerun impact on the 7 replaced approaches**

,run_id,old_friedman_avg_rank,corrected_friedman_avg_rank,delta_friedman_avg_rank,old_mean_score,corrected_mean_score,delta_mean_score
9,moelora_tr8_e2_k1,7.607143,6.571429,-1.035714,0.584838,0.587054,0.002216
12,mola_tr8_e2_k1,8.273810,7.547619,-0.726190,0.553305,0.545755,-0.007549
10,mola_tr4_e2_k1,8.523810,7.583333,-0.940476,0.554946,0.522345,-0.032601
3,lora_r4,12.904762,8.047619,-4.857143,0.118888,0.474582,0.355693
6,moelora_tr4_e2_k1,11.630952,8.071429,-3.559524,0.298313,0.476779,0.178466
13,mov_dense_e2,7.011905,8.416667,1.404762,0.632994,0.446768,-0.186226
2,hydralora_r4_heads2,10.952381,9.845238,-1.107143,0.356344,0.361034,0.004690


In [7]:
def best_by_kind(analysis):
    ranking = analysis["ranking"].sort_values(["kind", "friedman_avg_rank"])
    return ranking.groupby("kind", as_index=False).head(1)[["kind", "run_id", "friedman_avg_rank", "mean_score"]]


best_kind_comparison = best_by_kind(old_analysis).merge(
    best_by_kind(corrected_analysis),
    on="kind",
    how="outer",
    suffixes=("_old", "_corrected"),
)
best_kind_comparison["changed_best_config"] = best_kind_comparison.run_id_old != best_kind_comparison.run_id_corrected

old_top3 = old_analysis["overall"]["top3_configs"]
corrected_top3 = corrected_analysis["overall"]["top3_configs"]
promotion_status = ranking_comparison[ranking_comparison.run_id.isin(PROMOTED_TO_FINAL)][[
    "run_id", "old_friedman_avg_rank", "corrected_friedman_avg_rank", "old_mean_score", "corrected_mean_score"
]].sort_values("corrected_friedman_avg_rank")

decision_checks = pd.DataFrame([
    {"check": "top 3 configs from the original significance notebook remain the same", "pass": set(old_top3) == set(corrected_top3)},
    {"check": "no rerun approach enters the corrected top 3 configs", "pass": not bool(set(corrected_top3) & EXPECTED_RERUNS)},
    {"check": "best config per approach family/kind remains unchanged", "pass": not bool(best_kind_comparison.changed_best_config.any())},
    {"check": "final promoted 96/200-language run IDs were not among the 7 rerun targets", "pass": PROMOTED_TO_FINAL.isdisjoint(EXPECTED_RERUNS)},
])

display(Markdown(f"Old top 3 configs: `{old_top3}`"))
display(Markdown(f"Corrected top 3 configs: `{corrected_top3}`"))
display(best_kind_comparison)
display(Markdown("**Previously promoted final-stage run IDs**"))
display(promotion_status)
display(decision_checks)
assert decision_checks["pass"].all(), decision_checks.loc[~decision_checks["pass"], "check"].tolist()


Old top 3 configs: `['moelora_tr64_e32_k4', 'hydralora_r2_heads32', 'moelora_tr64_e32_k8']`

Corrected top 3 configs: `['moelora_tr64_e32_k4', 'hydralora_r2_heads32', 'moelora_tr64_e32_k8']`

,kind,run_id_old,friedman_avg_rank_old,mean_score_old,run_id_corrected,friedman_avg_rank_corrected,mean_score_corrected,changed_best_config
0,hydralora,hydralora_r2_heads32,4.238095,0.801961,hydralora_r2_heads32,5.583333,0.657946,False
1,lora,lora_r4,12.904762,0.118888,lora_r4,8.047619,0.474582,False
2,moelora,moelora_tr64_e32_k4,4.226190,0.789081,moelora_tr64_e32_k4,5.535714,0.633914,False
3,mola,mola_tr64_e32_k4,5.797619,0.721210,mola_tr64_e32_k4,7.190476,0.545069,False
4,movlora,mov_dense_e2,7.011905,0.632994,mov_dense_e2,8.416667,0.446768,False


**Previously promoted final-stage run IDs**

,run_id,old_friedman_avg_rank,corrected_friedman_avg_rank,old_mean_score,corrected_mean_score
7,moelora_tr64_e32_k4,4.226190,5.535714,0.789081,0.633914
1,hydralora_r2_heads32,4.238095,5.583333,0.801961,0.657946
11,mola_tr64_e32_k4,5.797619,7.190476,0.721210,0.545069


,check,pass
0,top 3 configs from the original significance n...,True
1,no rerun approach enters the corrected top 3 c...,True
2,best config per approach family/kind remains u...,True
3,final promoted 96/200-language run IDs were no...,True


## Final Conclusion

The original issue was real: 7 runs trained for more than one epoch

The expanded dataset now has 400k train sampled plus 7705 validation samples, preserves the intended language distribution, and has no old/new source-key or text-hash overlap according to the audit artifacts (no doubled data)

The 7 targeted approaches were rerun on the expanded dataset and all remain below one epoch

Replacing the old results with the rerun results does not change the top-3 outcome from the original significance notebook, does not change the best configuration per approach family, and does not require rerunning the already-promoted 96/200-language follow-up runs